# Phân loại ảnh bằng SIFT + Bag of Visual Words + SVM

Notebook này được thiết kế để tải mã nguồn từ GitHub và chạy trực tiếp trên Kaggle.

Trước khi chọn **Run All**:

1. Bật Internet trong phần **Notebook options** để Git và pip có thể hoạt động.
2. Chọn **Add Input** để gắn bộ dữ liệu Caltech-101 đã có cấu trúc thư mục ảnh vào notebook.
3. Nếu repository là private, cần đổi `REPO_URL` sang URL có thông tin xác thực phù hợp. Không ghi token trực tiếp vào notebook công khai.

## 1. Cấu hình

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/ngvihoa/bovw-image-classification.git"
BRANCH = "main"
PROJECT_DIR = Path("/kaggle/working/bovw-image-classification")
OUTPUT_DIR = Path("/kaggle/working/bovw_output")

# Giữ None để tự tìm thư mục chứa các lớp trong /kaggle/input.
# Hoặc nhập đường dẫn cụ thể, ví dụ:
# DATA_DIR = Path("/kaggle/input/caltech101/101_ObjectCategories")
DATA_DIR = None

CLASSES = ["airplanes", "Motorbikes", "Faces", "watch", "car_side"]
VOCAB_SIZE = 500
TEST_SAMPLE_INDEX = 0  # Vị trí ảnh test sẽ được hiển thị sau đánh giá.
SEED = 42

print("Project directory:", PROJECT_DIR)
print("Output directory :", OUTPUT_DIR)

## 2. Clone hoặc cập nhật repository

Ở lần chạy đầu tiên, cell này dùng `git clone`. Nếu thư mục dự án đã tồn tại do chạy lại notebook, cell sẽ dùng `git pull --ff-only`.

In [ ]:
import subprocess

def run_command(command, cwd=None):
    print("$", " ".join(map(str, command)))
    subprocess.run([str(item) for item in command], cwd=cwd, check=True)

if (PROJECT_DIR / ".git").is_dir():
    run_command(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR)
    run_command(["git", "checkout", BRANCH], cwd=PROJECT_DIR)
    run_command(["git", "pull", "--ff-only", "origin", BRANCH], cwd=PROJECT_DIR)
else:
    run_command(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, PROJECT_DIR])

run_command(["git", "log", "-1", "--oneline"], cwd=PROJECT_DIR)

## 3. Cài đặt thư viện

Kaggle thường đã có NumPy, scikit-learn và OpenCV. Cell này vẫn kiểm tra file `requirements.txt` của repository và cài các gói còn thiếu hoặc chưa đúng phiên bản.

In [ ]:
import sys

requirements_file = PROJECT_DIR / "requirements.txt"
if not requirements_file.is_file():
    raise FileNotFoundError(f"Không tìm thấy {requirements_file}")

run_command([sys.executable, "-m", "pip", "install", "-q", "-r", requirements_file])

## 4. Huấn luyện và đánh giá với K = 500

Cell này tự tìm thư mục dữ liệu trong `/kaggle/input` nếu `DATA_DIR` là `None`, sau đó sử dụng trực tiếp các mô-đun trong `src/` để chạy SIFT, BoVW và SVM với một kích thước từ điển duy nhất là K = 500.

In [ ]:
import os
import sys
import numpy as np

def find_data_directory(search_root):
    required = {name.casefold(): name for name in CLASSES}
    for current, directories, _files in os.walk(Path(search_root)):
        actual = {name.casefold(): name for name in directories}
        if set(required).issubset(actual):
            names = [actual[name.casefold()] for name in CLASSES]
            return Path(current), names
    return None, None

search_root = DATA_DIR if DATA_DIR is not None else Path("/kaggle/input")
DATA_DIR, detected_classes = find_data_directory(search_root)
if DATA_DIR is None:
    raise FileNotFoundError(
        f"Không tìm thấy thư mục chứa đủ các lớp {CLASSES} bên dưới {search_root}."
    )
CLASSES = detected_classes
print("Dữ liệu sử dụng:", DATA_DIR)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import config
from src.dataset import load_dataset
from src.sift import collect_training_descriptors
from src.vocabulary import build_vocabulary, save_vocabulary
from src.bovw import build_features, save_features
from src.classifier import train_classifier, save_model
from src.evaluate import compute_metrics, save_metrics

# Ghi kết quả vào /kaggle/working thay vì bên trong repository.
config.DATA_DIR = str(DATA_DIR)
config.CLASSES = list(CLASSES)
config.RANDOM_STATE = SEED
config.ARTIFACT_DIR = str(OUTPUT_DIR / "artifacts")
config.VOCAB_DIR = str(OUTPUT_DIR / "artifacts" / "vocabularies")
config.FEATURES_DIR = str(OUTPUT_DIR / "artifacts" / "features")
config.CLASSIFIERS_DIR = str(OUTPUT_DIR / "artifacts" / "classifiers")
config.RESULTS_DIR = str(OUTPUT_DIR / "results")
config.METRICS_DIR = str(OUTPUT_DIR / "results" / "metrics")

train_paths, test_paths, train_labels, test_labels = load_dataset(
    data_dir=str(DATA_DIR),
    classes=CLASSES,
    test_size=config.TEST_SIZE,
    random_state=SEED,
)
print(f"Train: {len(train_paths)} ảnh | Test: {len(test_paths)} ảnh")

print(f"\n{'=' * 70}\nBắt đầu huấn luyện với K={VOCAB_SIZE}")
np.random.seed(SEED)

descriptors = collect_training_descriptors(
    train_paths,
    max_descriptor_per_image=config.MAX_DESCRIPTOR_PER_IMAGE,
    max_total_descriptors=config.MAX_TOTAL_DESCRIPTORS,
)
vocabulary = build_vocabulary(descriptors, vocab_size=VOCAB_SIZE)
save_vocabulary(vocabulary, vocab_size=VOCAB_SIZE)

train_features = build_features(train_paths, vocabulary, vocab_size=VOCAB_SIZE)
test_features = build_features(test_paths, vocabulary, vocab_size=VOCAB_SIZE)
save_features(train_features, "train", vocab_size=VOCAB_SIZE)
save_features(test_features, "test", vocab_size=VOCAB_SIZE)

classifier = train_classifier(train_features, train_labels)
save_model(classifier, vocab_size=VOCAB_SIZE)
predictions = classifier.predict(test_features)
metrics = compute_metrics(test_labels, predictions)
save_metrics(metrics, vocab_size=VOCAB_SIZE)

print(f"K={VOCAB_SIZE}: {metrics}")
print("Đã hoàn tất. Kết quả tại:", OUTPUT_DIR)

## 5. Kết quả đánh giá

In [ ]:
import json

metrics_path = OUTPUT_DIR / "results" / "metrics" / f"metrics_{VOCAB_SIZE}.json"
with metrics_path.open(encoding="utf-8") as file:
    saved_metrics = json.load(file)

print(f"K              : {VOCAB_SIZE}")
print(f"Accuracy       : {saved_metrics['accuracy']:.4f}")
print(f"Macro precision: {saved_metrics['precision']:.4f}")
print(f"Macro recall   : {saved_metrics['recall']:.4f}")
print(f"Macro F1-score : {saved_metrics['f1_score']:.4f}")

## 6. Hiển thị kết quả trên một ảnh test

Cell dưới đây hiển thị một ảnh thuộc tập kiểm tra cùng nhãn thật, nhãn dự đoán và trạng thái đúng/sai. Có thể đổi `TEST_SAMPLE_INDEX` trong cell cấu hình để xem ảnh khác.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

sample_index = TEST_SAMPLE_INDEX % len(test_paths)
image_path = Path(test_paths[sample_index])
true_label = CLASSES[int(test_labels[sample_index])]
predicted_label = CLASSES[int(predictions[sample_index])]
is_correct = true_label == predicted_label

image = Image.open(image_path).convert("RGB")
plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.axis("off")
plt.title(
    f"Nhãn thật: {true_label} | Dự đoán: {predicted_label} | "
    f"{'ĐÚNG' if is_correct else 'SAI'}",
    color="green" if is_correct else "red",
)
plt.show()
print("Ảnh test:", image_path)